# 스마트 창고 출고 지연 예측 — Model v4b (DL Improved)

**v4 대비 개선 사항:**
- MLP / LSTM 에 `StandardScaler` 추가 (미적용이 성능 저하의 핵심 원인)
- LSTM 시퀀스 생성 로직 벡터화 (O(n²) → O(n))
- TabNet lr 2e-3 → 1e-3, n_d/n_a 64 → 128
- MLP CosineAnnealing 스케줄러로 교체
- LSTM fold OOF 매핑 로직 재작성

**실행 환경**: Google Colab T4 GPU

## 0. 환경 설정

In [ ]:
!pip install -q pytorch-tabnet optuna lightgbm xgboost catboost

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import lightgbm as lgb
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from pytorch_tabnet.tab_model import TabNetRegressor
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.impute import SimpleImputer
import optuna
from optuna.samplers import TPESampler
import warnings
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

DEVICE  = 'cuda' if torch.cuda.is_available() else 'cpu'
SEED    = 42
N_FOLD  = 5
N_STEPS = 25
TARGET  = 'avg_delay_minutes_next_30m'
ID_COLS = ['ID', 'layout_id', 'scenario_id']

torch.manual_seed(SEED)
np.random.seed(SEED)

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

print(f'Device: {DEVICE}')
print(f'GPU: {torch.cuda.get_device_name(0) if DEVICE=="cuda" else "없음"}')

## 1. Google Drive 마운트 & 데이터 로드

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/warehouse/'

train  = pd.read_csv(DATA_DIR + 'train.csv')
test   = pd.read_csv(DATA_DIR + 'test.csv')
layout = pd.read_csv(DATA_DIR + 'layout_info.csv')

print(f'train: {train.shape} / test: {test.shape}')

## 2. v2 최적 파라미터 입력

In [ ]:
# ─── v2 Optuna best params 붙여넣기 ────────────────────────────
lgb_best = {'n_estimators': 700, 'learning_rate': 0.011479, 'max_depth': 7,
            'num_leaves': 157, 'subsample': 0.9417, 'colsample_bytree': 0.7183,
            'reg_alpha': 0.00231, 'reg_lambda': 0.00710, 'min_child_samples': 70,
            'random_state': SEED, 'verbose': -1, 'n_jobs': -1}

xgb_best = {'n_estimators': 1700, 'learning_rate': 0.010069, 'max_depth': 8,
            'subsample': 0.7702, 'colsample_bytree': 0.6282, 'reg_alpha': 0.00682,
            'reg_lambda': 5.9363, 'min_child_weight': 17, 'gamma': 1.2170,
            'random_state': SEED, 'verbosity': 0, 'n_jobs': -1,
            'tree_method': 'hist', 'device': 'cuda',
            'early_stopping_rounds': 100}

cat_best = {'iterations': 2000, 'learning_rate': 0.011497, 'depth': 10,
            'l2_leaf_reg': 5.1253, 'bagging_temperature': 0.0486,
            'random_strength': 0.00217, 'border_count': 56,
            'random_seed': SEED, 'verbose': 0,
            'early_stopping_rounds': 100, 'task_type': 'GPU'}
# ────────────────────────────────────────────────────────────────
print('파라미터 설정 완료')

## 3. 전처리 (v2 동일)

In [ ]:
train = train.merge(layout, on='layout_id', how='left')
test  = test.merge(layout,  on='layout_id', how='left')

le = LabelEncoder()
train['layout_type'] = le.fit_transform(train['layout_type'].astype(str))
test['layout_type']  = le.transform(test['layout_type'].astype(str))

train = train.sort_values(['scenario_id', 'shift_hour']).reset_index(drop=True)
test  = test.sort_values(['scenario_id',  'shift_hour']).reset_index(drop=True)

ts_features = [
    'order_inflow_15m', 'robot_utilization', 'congestion_score',
    'fault_count_15m',  'loading_dock_util', 'battery_mean',
    'blocked_path_15m', 'task_reassign_15m', 'low_battery_ratio',
    'avg_trip_distance',
]

def make_ts_features(df, cols, group_col='scenario_id'):
    df = df.copy()
    grp = df.groupby(group_col)
    for col in cols:
        for lag in [1, 2, 3]:
            df[f'{col}_lag{lag}'] = grp[col].shift(lag)
        for win in [3, 5]:
            df[f'{col}_roll_mean{win}'] = grp[col].transform(
                lambda x: x.shift(1).rolling(win, min_periods=1).mean())
            df[f'{col}_roll_std{win}'] = grp[col].transform(
                lambda x: x.shift(1).rolling(win, min_periods=2).std())
        df[f'{col}_cumsum'] = grp[col].transform(lambda x: x.shift(1).cumsum())
    return df

def make_extra_features(df):
    df = df.copy()
    df['time_step']          = df.groupby('scenario_id').cumcount()
    df['robot_active_ratio'] = df['robot_active'] / (
        df['robot_active'] + df['robot_idle'] + df['robot_charging'] + 1e-6)
    df['battery_congestion'] = df['low_battery_ratio'] * df['congestion_score']
    df['urgent_volume']      = df['urgent_order_ratio'] * df['order_inflow_15m']
    df['fault_impact']       = df['fault_count_15m'] * df['avg_recovery_time']
    df['dock_stress']        = df['outbound_truck_wait_min'] * df['loading_dock_util']
    return df

train = make_ts_features(train, ts_features)
test  = make_ts_features(test,  ts_features)
train = make_extra_features(train)
test  = make_extra_features(test)

feat_cols = [c for c in train.columns if c not in ID_COLS + [TARGET]]
X         = train[feat_cols]
y         = train[TARGET]
X_test    = test[feat_cols]
groups    = train['scenario_id']

print(f'피처 수: {len(feat_cols)}')

## 4. GBDT 재학습 → OOF 생성 (v2 params)

In [ ]:
gkf5 = GroupKFold(n_splits=N_FOLD)

def run_gbdt_cv(model_fn, model_name, fit_kwargs_fn=None):
    oof   = np.zeros(len(X))
    preds = np.zeros(len(X_test))
    for fold, (tr_idx, val_idx) in enumerate(gkf5.split(X, y, groups)):
        X_tr, y_tr   = X.iloc[tr_idx], y.iloc[tr_idx]
        X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
        model = model_fn()
        kw = fit_kwargs_fn(X_val, y_val) if fit_kwargs_fn else {}
        model.fit(X_tr, y_tr, **kw)
        oof[val_idx] = model.predict(X_val)
        preds += model.predict(X_test) / N_FOLD
        print(f'  [{model_name}] Fold {fold+1}  RMSE: {rmse(y_val, oof[val_idx]):.4f}')
    print(f'  [{model_name}] OOF RMSE: {rmse(y, oof):.4f}\n')
    return oof, preds, rmse(y, oof)

print('=== LightGBM ===')
lgb_oof, lgb_pred, lgb_score = run_gbdt_cv(
    lambda: LGBMRegressor(**lgb_best), 'LGB',
    lambda xv, yv: {'eval_set': [(xv, yv)],
                    'callbacks': [lgb.early_stopping(100, verbose=False),
                                  lgb.log_evaluation(500)]})

In [ ]:
print('=== XGBoost (GPU) ===')
xgb_oof, xgb_pred, xgb_score = run_gbdt_cv(
    lambda: XGBRegressor(**xgb_best), 'XGB',
    lambda xv, yv: {'eval_set': [(xv, yv)], 'verbose': False})

In [ ]:
print('=== CatBoost (GPU) ===')
cat_oof, cat_pred, cat_score = run_gbdt_cv(
    lambda: CatBoostRegressor(**cat_best), 'CAT',
    lambda xv, yv: {'eval_set': (xv, yv)})

## 5. DL 전처리 — NaN 처리 + StandardScaler

> **핵심 수정**: v4에서 누락됐던 StandardScaler 추가  
> MLP/LSTM은 피처 스케일에 민감하여 미적용 시 성능이 크게 저하됨

In [ ]:
# 1) NaN → median 대체
imputer  = SimpleImputer(strategy='median')
X_imp      = imputer.fit_transform(X)
X_test_imp = imputer.transform(X_test)

# 2) StandardScaler 정규화
scaler     = StandardScaler()
X_scaled      = scaler.fit_transform(X_imp).astype(np.float32)
X_test_scaled = scaler.transform(X_test_imp).astype(np.float32)

y_arr = y.values.astype(np.float32)

print(f'스케일링 완료 | mean≈0: {X_scaled.mean():.4f} | std≈1: {X_scaled.std():.4f}')

## 6. TabNet → OOF 생성 (개선)

In [ ]:
tabnet_oof  = np.zeros(len(X_scaled))
tabnet_pred = np.zeros(len(X_test_scaled))

tabnet_params = dict(
    n_d=128, n_a=128,          # v4: 64 → 128 (표현력 향상)
    n_steps=5, gamma=1.3,
    n_independent=2, n_shared=2,
    optimizer_fn=torch.optim.Adam,
    optimizer_params={'lr': 1e-3, 'weight_decay': 1e-5},   # v4: 2e-3 → 1e-3
    scheduler_fn=torch.optim.lr_scheduler.StepLR,
    scheduler_params={'step_size': 20, 'gamma': 0.8},
    mask_type='entmax',
    device_name=DEVICE,
    seed=SEED,
    verbose=0,
)

for fold, (tr_idx, val_idx) in enumerate(gkf5.split(X_scaled, y_arr, groups)):
    X_tr, y_tr   = X_scaled[tr_idx], y_arr[tr_idx].reshape(-1, 1)
    X_val, y_val = X_scaled[val_idx], y_arr[val_idx].reshape(-1, 1)

    model = TabNetRegressor(**tabnet_params)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        eval_name=['val'], eval_metric=['rmse'],
        max_epochs=200, patience=20,
        batch_size=4096, virtual_batch_size=512,
    )

    tabnet_oof[val_idx] = model.predict(X_val).flatten()
    tabnet_pred        += model.predict(X_test_scaled).flatten() / N_FOLD

    print(f'  [TabNet] Fold {fold+1}  RMSE: {rmse(y_arr[val_idx], tabnet_oof[val_idx]):.4f}')

tabnet_score = rmse(y_arr, tabnet_oof)
print(f'  [TabNet] OOF RMSE: {tabnet_score:.4f}')

## 7. MLP (ResNet 스타일) → OOF 생성 (개선)

In [ ]:
class ResBlock(nn.Module):
    def __init__(self, dim, dropout):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(dim, dim), nn.BatchNorm1d(dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(dim, dim), nn.BatchNorm1d(dim),
        )
        self.act = nn.ReLU()

    def forward(self, x):
        return self.act(x + self.block(x))


class MLP(nn.Module):
    def __init__(self, input_dim, hidden=512, n_blocks=4, dropout=0.3):
        super().__init__()
        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, hidden), nn.BatchNorm1d(hidden), nn.ReLU()
        )
        self.blocks = nn.Sequential(*[ResBlock(hidden, dropout) for _ in range(n_blocks)])
        self.head   = nn.Linear(hidden, 1)

    def forward(self, x):
        return self.head(self.blocks(self.input_proj(x))).squeeze(-1)


class TabDataset(Dataset):
    def __init__(self, X, y=None):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y) if y is not None else None

    def __len__(self): return len(self.X)

    def __getitem__(self, i):
        return (self.X[i], self.y[i]) if self.y is not None else self.X[i]


def train_mlp(X_tr, y_tr, X_val, y_val, input_dim, epochs=150, patience=20):
    model = MLP(input_dim, hidden=512, n_blocks=4).to(DEVICE)
    opt   = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
    # CosineAnnealing: 부드러운 lr 감소로 안정적 수렴
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    crit  = nn.MSELoss()

    tr_loader  = DataLoader(TabDataset(X_tr, y_tr),   batch_size=4096, shuffle=True)
    val_loader = DataLoader(TabDataset(X_val, y_val), batch_size=4096)

    best_val, best_state, no_imp = np.inf, None, 0
    for epoch in range(epochs):
        model.train()
        for xb, yb in tr_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            crit(model(xb), yb).backward()
            opt.step()
        sched.step()

        model.eval()
        with torch.no_grad():
            val_preds = torch.cat(
                [model(xb.to(DEVICE)) for xb, _ in val_loader]
            ).cpu().numpy()
        val_rmse = rmse(y_val, val_preds)

        if val_rmse < best_val:
            best_val, best_state, no_imp = val_rmse, model.state_dict(), 0
        else:
            no_imp += 1
            if no_imp >= patience:
                break

    model.load_state_dict(best_state)
    return model


def predict_mlp(model, X):
    model.eval()
    loader = DataLoader(TabDataset(X), batch_size=4096)
    with torch.no_grad():
        return torch.cat([model(xb.to(DEVICE)) for xb in loader]).cpu().numpy()


print('MLP 클래스 정의 완료')

In [ ]:
mlp_oof  = np.zeros(len(X_scaled))
mlp_pred = np.zeros(len(X_test_scaled))
input_dim = X_scaled.shape[1]

for fold, (tr_idx, val_idx) in enumerate(gkf5.split(X_scaled, y_arr, groups)):
    model = train_mlp(
        X_scaled[tr_idx],  y_arr[tr_idx],
        X_scaled[val_idx], y_arr[val_idx],
        input_dim
    )
    mlp_oof[val_idx]  = predict_mlp(model, X_scaled[val_idx])
    mlp_pred         += predict_mlp(model, X_test_scaled) / N_FOLD

    print(f'  [MLP] Fold {fold+1}  RMSE: {rmse(y_arr[val_idx], mlp_oof[val_idx]):.4f}')

mlp_score = rmse(y_arr, mlp_oof)
print(f'  [MLP] OOF RMSE: {mlp_score:.4f}')

## 8. LSTM → OOF 생성 (개선)

핵심 수정:
- `to_sequences` 벡터화 (O(n²) → O(n))
- fold OOF 매핑 로직 재작성
- 스케일된 피처 사용

In [ ]:
class WarehouseLSTM(nn.Module):
    def __init__(self, input_size, hidden=256, n_layers=2, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden, n_layers,
                            batch_first=True, dropout=dropout)
        self.head = nn.Sequential(
            nn.Linear(hidden, 64), nn.ReLU(), nn.Linear(64, 1)
        )

    def forward(self, x):                      # (batch, 25, features)
        out, _ = self.lstm(x)                  # (batch, 25, hidden)
        return self.head(out).squeeze(-1)      # (batch, 25)


class SeqDataset(Dataset):
    def __init__(self, X_seq, y_seq=None):
        self.X = torch.FloatTensor(X_seq)
        self.y = torch.FloatTensor(y_seq) if y_seq is not None else None

    def __len__(self): return len(self.X)

    def __getitem__(self, i):
        return (self.X[i], self.y[i]) if self.y is not None else self.X[i]


def to_sequences_fast(X_flat, y_flat, scenario_arr, step_arr, unique_sc):
    """벡터화된 시퀀스 변환 O(n)"""
    sc_to_idx = {sc: i for i, sc in enumerate(unique_sc)}
    sc_idx    = np.array([sc_to_idx[sc] for sc in scenario_arr])
    step_idx  = step_arr.astype(int)

    n, f = len(unique_sc), X_flat.shape[1]
    X_seq = np.zeros((n, N_STEPS, f), dtype=np.float32)
    X_seq[sc_idx, step_idx] = X_flat

    y_seq = None
    if y_flat is not None:
        y_seq = np.zeros((n, N_STEPS), dtype=np.float32)
        y_seq[sc_idx, step_idx] = y_flat

    return X_seq, y_seq


def train_lstm(X_seq_tr, y_seq_tr, X_seq_val, y_seq_val, input_dim,
               epochs=80, patience=12):
    model = WarehouseLSTM(input_dim).to(DEVICE)
    opt   = torch.optim.Adam(model.parameters(), lr=5e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    crit  = nn.MSELoss()

    tr_loader  = DataLoader(SeqDataset(X_seq_tr,  y_seq_tr),  batch_size=256, shuffle=True)
    val_loader = DataLoader(SeqDataset(X_seq_val, y_seq_val), batch_size=256)

    best_val, best_state, no_imp = np.inf, None, 0
    for epoch in range(epochs):
        model.train()
        for xb, yb in tr_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            crit(model(xb), yb).backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        sched.step()

        model.eval()
        with torch.no_grad():
            val_preds = torch.cat(
                [model(xb.to(DEVICE)) for xb, _ in val_loader]
            ).cpu().numpy()   # (n_val_sc, 25)
        val_rmse = np.sqrt(np.mean((val_preds - y_seq_val) ** 2))

        if val_rmse < best_val:
            best_val, best_state, no_imp = val_rmse, model.state_dict(), 0
        else:
            no_imp += 1
            if no_imp >= patience:
                break

    model.load_state_dict(best_state)
    return model


def predict_seq(model, X_seq):
    model.eval()
    loader = DataLoader(SeqDataset(X_seq), batch_size=256)
    with torch.no_grad():
        return torch.cat([model(xb.to(DEVICE)) for xb in loader]).cpu().numpy()


print('LSTM 클래스 정의 완료')

In [ ]:
# 시나리오별 step 인덱스 (cumcount 기반)
train_scenarios = train['scenario_id'].values
test_scenarios  = test['scenario_id'].values
train_steps     = train.groupby('scenario_id').cumcount().values
test_steps      = test.groupby('scenario_id').cumcount().values

unique_train_sc = pd.unique(train['scenario_id'])
unique_test_sc  = pd.unique(test['scenario_id'])

# 전체 시퀀스 배열 생성
X_seq_all, y_seq_all = to_sequences_fast(
    X_scaled, y_arr, train_scenarios, train_steps, unique_train_sc)
X_seq_test, _        = to_sequences_fast(
    X_test_scaled, None, test_scenarios, test_steps, unique_test_sc)

print(f'train seq: {X_seq_all.shape}')   # (10000, 25, features)
print(f'test  seq: {X_seq_test.shape}')  # (2000,  25, features)

In [ ]:
lstm_oof  = np.zeros(len(X_scaled))
lstm_pred = np.zeros(len(X_test_scaled))
input_dim = X_scaled.shape[1]
n_sc      = len(unique_train_sc)

gkf_sc = GroupKFold(n_splits=N_FOLD)

for fold, (tr_sc_idx, val_sc_idx) in enumerate(
    gkf_sc.split(range(n_sc), range(n_sc), range(n_sc))
):
    # 시나리오 인덱스 → 행 인덱스 매핑
    val_sc_set  = set(unique_train_sc[val_sc_idx])
    val_row_idx = np.where(np.isin(train_scenarios, list(val_sc_set)))[0]

    model = train_lstm(
        X_seq_all[tr_sc_idx],  y_seq_all[tr_sc_idx],
        X_seq_all[val_sc_idx], y_seq_all[val_sc_idx],
        input_dim
    )

    # 시나리오 단위 예측 → 행 단위 OOF 복원
    val_pred_seq = predict_seq(model, X_seq_all[val_sc_idx])  # (n_val_sc, 25)
    sc_to_seq    = {sc: i for i, sc in enumerate(unique_train_sc[val_sc_idx])}
    for row_i in val_row_idx:
        sc   = train_scenarios[row_i]
        step = train_steps[row_i]
        lstm_oof[row_i] = val_pred_seq[sc_to_seq[sc], step]

    lstm_pred += predict_seq(model, X_seq_test).flatten() / N_FOLD

    score = rmse(y_arr[val_row_idx], lstm_oof[val_row_idx])
    print(f'  [LSTM] Fold {fold+1}  RMSE: {score:.4f}')

lstm_score = rmse(y_arr, lstm_oof)
print(f'  [LSTM] OOF RMSE: {lstm_score:.4f}')

## 9. Ridge 스태킹 + Optuna 앙상블

In [ ]:
model_names_meta = ['LGB', 'XGB', 'CAT', 'TabNet', 'MLP', 'LSTM']
all_oofs  = [lgb_oof,  xgb_oof,  cat_oof,  tabnet_oof,  mlp_oof,  lstm_oof]
all_preds = [lgb_pred, xgb_pred, cat_pred, tabnet_pred, mlp_pred, lstm_pred]
all_scores = [lgb_score, xgb_score, cat_score, tabnet_score, mlp_score, lstm_score]

print('모델별 OOF RMSE')
print('-' * 35)
for name, score in zip(model_names_meta, all_scores):
    delta = score - 21.2717
    print(f'  {name:8s}: {score:.4f}  ({delta:+.4f} vs v2)')

In [ ]:
# Ridge 스태킹
meta_train = np.column_stack(all_oofs)
meta_test  = np.column_stack(all_preds)

ridge_oof  = np.zeros(len(meta_train))
ridge_pred = np.zeros(len(meta_test))

for fold, (tr_idx, val_idx) in enumerate(gkf5.split(meta_train, y_arr, groups)):
    meta = Ridge(alpha=1.0)
    meta.fit(meta_train[tr_idx], y_arr[tr_idx])
    ridge_oof[val_idx] = meta.predict(meta_train[val_idx])
    ridge_pred        += meta.predict(meta_test) / N_FOLD

ridge_score = rmse(y_arr, ridge_oof)
print(f'Ridge 스태킹 OOF RMSE: {ridge_score:.4f}  ({ridge_score - 21.2717:+.4f} vs v2)')

In [ ]:
# Optuna 앙상블 가중치 최적화
def ens_objective(trial):
    ws    = [trial.suggest_float(f'w{i}', 0.0, 1.0) for i in range(6)]
    total = sum(ws)
    if total < 1e-6: return 1e9
    ws  = [w / total for w in ws]
    ens = sum(w * oof for w, oof in zip(ws, all_oofs))
    return rmse(y_arr, ens)

ens_study = optuna.create_study(direction='minimize', sampler=TPESampler(seed=SEED))
ens_study.optimize(ens_objective, n_trials=1000, show_progress_bar=True)

bw    = ens_study.best_params
total = sum(bw.values())
opt_w    = [bw[f'w{i}'] / total for i in range(6)]
opt_oof  = sum(w * oof  for w, oof  in zip(opt_w, all_oofs))
opt_pred = sum(w * pred for w, pred in zip(opt_w, all_preds))
opt_score = rmse(y_arr, opt_oof)

print('\n최적 가중치:')
for name, w in zip(model_names_meta, opt_w):
    print(f'  {name:8s}: {w:.4f}')
print(f'\nOptuna 앙상블 OOF RMSE: {opt_score:.4f}  ({opt_score - 21.2717:+.4f} vs v2)')

## 10. 최종 성능 비교

In [ ]:
import matplotlib.pyplot as plt

results = {
    'v2 GBDT (로컬 최선)': 21.2717,
    'v4 Ridge 스태킹':     21.2521,
    'v4b LGB': lgb_score, 'v4b XGB': xgb_score, 'v4b CAT': cat_score,
    'v4b TabNet': tabnet_score, 'v4b MLP': mlp_score, 'v4b LSTM': lstm_score,
    'v4b Ridge 스태킹': ridge_score,
    'v4b Optuna 앙상블': opt_score,
}

print('=' * 55)
print('최종 성능 비교')
print('=' * 55)
for name, score in results.items():
    delta = score - 21.2717
    mark  = '✓' if delta < 0 else ' '
    print(f'{mark} {name:28s}: {score:.4f}  ({delta:+.4f})')

fig, ax = plt.subplots(figsize=(13, 6))
colors  = ['lightgray', 'lightsteelblue',
           'steelblue', 'seagreen', 'coral',
           'orange', 'mediumpurple', 'hotpink',
           'gold', 'tomato']
bars = ax.barh(list(results.keys()), list(results.values()),
               color=colors, edgecolor='white')
for bar, score in zip(bars, results.values()):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
            f'{score:.4f}', va='center', fontsize=9)
ax.axvline(21.2717, color='steelblue', linestyle='--', lw=1.5, label='v2 기준선')
ax.set_xlabel('OOF RMSE')
ax.set_title('v4b DL Improved vs 이전 버전', fontsize=13)
ax.legend()
plt.tight_layout()
plt.show()

## 11. 제출 파일 생성

In [ ]:
candidates = [
    (ridge_score, ridge_pred, 'Ridge스태킹'),
    (opt_score,   opt_pred,   'Optuna앙상블'),
]
best_score, best_pred, best_label = min(candidates, key=lambda x: x[0])

submission = pd.read_csv(DATA_DIR + 'sample_submission.csv')
submission[TARGET] = best_pred
submission.to_csv(DATA_DIR + 'submission_v4b.csv', index=False)

print(f'submission_v4b.csv 저장 완료')
print(f'선택된 앙상블 : {best_label}  (RMSE: {best_score:.4f})')
print(f'v2 대비       : {21.2717 - best_score:+.4f}')
print(submission[TARGET].describe().round(4))